# SurrogateNN DSGE on Google Colab: JAX, NumPyro, Gemini

This notebook is a Colab-ready pilot for the Python/JAX translation. It has three layers:

1. A self-contained JAX/NumPyro smoke run that does not require the Julia repository.
2. An optional full Julia CPU vs Python/JAX benchmark, if both GitHub repositories are reachable from Colab.
3. An optional Gemini summary cell using the official Google Gen AI Python SDK.

Use `Runtime -> Change runtime type -> GPU` before running. The notebook prints `jax.devices()` so you can verify whether Colab actually exposed a GPU to JAX.

## Configuration

Set `REPO_URL` to the GitHub URL for the Python translation repo before running on Colab. Keep `RUN_FULL_JULIA_BENCHMARK = False` until the self-contained smoke cells pass. If you change the JAX CUDA wheel after importing JAX, restart the Colab runtime before re-running the install cell.

In [ ]:
REPO_URL = "https://github.com/matyasfarkas/SurrogateNN_DSGE.git"
BRANCH = "main"

# JAX install choice. Use "auto", "cuda13", or "cuda12" for GPU. Use "cpu" only as a fallback.
# The notebook defaults to GPU-first and fails loudly if Colab does not expose a JAX GPU.
JAX_BACKEND = "auto"
STRICT_GPU = True
FORCE_REINSTALL_JAX = True
UPDATE_EXISTING_CLONE = True

# Optional full benchmark. This requires the Julia source repo to be cloneable from Colab.
RUN_FULL_JULIA_BENCHMARK = False
JULIA_REPO_URL = "https://github.com/<your-user-or-org>/SurrogateNN_Estimation.jl.git"

# Optional Gemini analysis. Store GOOGLE_API_KEY in Colab secrets or set it as an env var.
USE_GEMINI = False
GEMINI_MODEL = "gemini-2.5-flash"  # Change if your Google AI Studio account exposes a different model.

# Keep this small first. Increase only after the smoke path is stable.
NUTS_WARMUP = 8
NUTS_SAMPLES = 8
TIMING_REPS = 20

## Install and import

This cell clones or updates the Python repo, selects a CPU or CUDA JAX wheel before importing JAX, installs runtime dependencies explicitly, and then installs this package with `--no-deps` so package metadata cannot downgrade or replace the selected JAX wheel. If this cell has already imported JAX in the current runtime, restart the Colab runtime before changing JAX backend options.

In [ ]:
import os

# These must be set before importing JAX. CPU mode prevents CUDA initialization;
# GPU mode clears CPU-forcing variables and LD_LIBRARY_PATH so JAX pip CUDA wheels win.
jax_backend_requested = globals().get("JAX_BACKEND", "auto")
if jax_backend_requested == "cpu":
    os.environ["JAX_PLATFORMS"] = "cpu"
    os.environ["JAX_PLATFORM_NAME"] = "cpu"
else:
    os.environ.pop("JAX_PLATFORMS", None)
    os.environ.pop("JAX_PLATFORM_NAME", None)
    removed_ld_library_path = os.environ.pop("LD_LIBRARY_PATH", None)
    if removed_ld_library_path:
        print("Cleared LD_LIBRARY_PATH so JAX uses pip-installed CUDA libraries.")

import re
import sys
import subprocess
import time
from pathlib import Path

def run(cmd, cwd=None, check=True):
    print("$", " ".join(map(str, cmd)))
    return subprocess.run(cmd, cwd=cwd, check=check, text=True)

def run_text(cmd, cwd=None, check=False):
    print("$", " ".join(map(str, cmd)))
    proc = subprocess.run(cmd, cwd=cwd, check=check, text=True, capture_output=True)
    text = (proc.stdout or "") + (proc.stderr or "")
    print(text)
    return text

if "jax" in sys.modules:
    raise RuntimeError(
        "JAX is already imported in this runtime. Restart the Colab runtime before "
        "changing or reinstalling CUDA-enabled JAX wheels."
    )

ROOT = Path("/content/SurrogateNN_DSGE")
if not ROOT.exists():
    if "<your-user-or-org>" in REPO_URL:
        raise RuntimeError("Set REPO_URL to the Python translation repo before running on Colab.")
    run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(ROOT)])
elif UPDATE_EXISTING_CLONE:
    run(["git", "fetch", "origin", BRANCH], cwd=ROOT)
    run(["git", "checkout", BRANCH], cwd=ROOT)
    run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=ROOT)

os.chdir(ROOT)
print("Working directory:", Path.cwd())

nvidia_smi = run_text(["bash", "-lc", "nvidia-smi || true"])
cuda_major = None
match = re.search(r"CUDA Version:\\s*([0-9]+)", nvidia_smi)
if match:
    cuda_major = int(match.group(1))
driver_match = re.search(r"Driver Version:\\s*([0-9.]+)", nvidia_smi)
driver_version = driver_match.group(1) if driver_match else "unknown"
print("Detected NVIDIA driver:", driver_version, "CUDA major:", cuda_major)

if JAX_BACKEND == "auto":
    if cuda_major is None:
        if STRICT_GPU:
            raise RuntimeError("STRICT_GPU=True but nvidia-smi did not report a CUDA runtime. Set Colab runtime type to GPU.")
        selected_jax_extra = "cpu"
    elif cuda_major >= 13:
        selected_jax_extra = "cuda13"
    else:
        selected_jax_extra = "cuda12"
else:
    selected_jax_extra = JAX_BACKEND
if selected_jax_extra not in {"cpu", "cuda12", "cuda13"}:
    raise ValueError(f"Unsupported JAX_BACKEND={JAX_BACKEND!r}.")
if STRICT_GPU and selected_jax_extra == "cpu":
    raise RuntimeError("STRICT_GPU=True but selected_jax_extra='cpu'. Set JAX_BACKEND to auto/cuda13/cuda12 and use a GPU runtime.")
print("Selected JAX install target:", selected_jax_extra)

run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
if FORCE_REINSTALL_JAX:
    run(
        [
            sys.executable,
            "-m",
            "pip",
            "uninstall",
            "-y",
            "jax",
            "jaxlib",
            "jax-cuda12-pjrt",
            "jax-cuda12-plugin",
            "jax-cuda13-pjrt",
            "jax-cuda13-plugin",
        ],
        check=False,
    )

jax_package = "jax" if selected_jax_extra == "cpu" else f"jax[{selected_jax_extra}]"
run([sys.executable, "-m", "pip", "install", "--upgrade", "--no-cache-dir", jax_package])

# Install runtime deps explicitly, then install this repo without dependency resolution so JAX is not downgraded.
run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "sympy>=1.13,<2",
        "scipy>=1.10,<2",
        "numpyro>=0.19,<0.20",
        "pytest>=8.0,<9",
        "google-genai>=1.0",
    ]
)
run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"])

import jax
import jaxlib
import jax.numpy as jnp
import numpy as np
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS

from surrogatenn_dsge import (
    assemble_parameter_vector,
    build_linear_state_space_from_model,
    build_numpyro_kalman_model_jax,
    evaluate_numpyro_kalman_log_density_jax,
    kalman_loglikelihood_from_model_jax,
    parse_macro_model,
    simulate_linear_gaussian_state_space,
    solve_first_order_model,
)

print("Python:", sys.version)
print("JAX:", jax.__version__)
print("JAXLIB:", jaxlib.__version__)
print("NumPyro:", numpyro.__version__)
print("JAX devices:", jax.devices())
print("Default backend:", jax.default_backend())
gpu_devices = [device for device in jax.devices() if device.platform in {"gpu", "cuda"}]
if selected_jax_extra != "cpu" and not gpu_devices:
    raise RuntimeError("CUDA JAX was requested, but JAX did not report a GPU device. Check Colab GPU runtime, driver, and pip install output above.")
if selected_jax_extra == "cpu":
    assert jax.default_backend() == "cpu", f"Expected CPU backend, got {jax.default_backend()}"
else:
    assert jax.default_backend() in {"gpu", "cuda"}, f"Expected GPU backend, got {jax.default_backend()}"
    _gpu_probe = (jnp.ones((256, 256), dtype=jnp.float32) @ jnp.ones((256, 256), dtype=jnp.float32)).block_until_ready()
    _gpu_probe_devices = _gpu_probe.devices() if hasattr(_gpu_probe, "devices") else {getattr(_gpu_probe, "device", "unknown")}
    print("GPU probe result:", float(_gpu_probe[0, 0]), "devices:", _gpu_probe_devices)

In [ ]:
# Hardware sanity check. This is expected to fail harmlessly on CPU runtimes.
run(["bash", "-lc", "nvidia-smi || true"], check=False)
print("JAX devices:", jax.devices())
print("Default backend:", jax.default_backend())
try:
    import torch
    print("Torch CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("Torch CUDA device:", torch.cuda.get_device_name(0))
except Exception as exc:
    print("Torch CUDA diagnostic unavailable:", repr(exc))
run([sys.executable, "-m", "pip", "show", "jax", "jaxlib", "jax-cuda12-plugin", "jax-cuda13-plugin"], check=False)

## Self-contained DSGE smoke test

This uses a small MacroModelling-style model string, solves the first-order system, builds synthetic observations, JIT-compiles a Kalman likelihood, evaluates a NumPyro log density, and runs a tiny NUTS chain. This is the fastest way to confirm Colab + JAX + NumPyro is functional before running the heavier benchmark.

In [ ]:
MODEL_SOURCE = """
@model colab_likelihood_demo begin
    a[0] = rho_a * a[-1] + (1 - rho_a) * a_bar + eps_a[x]
    y[0] = rho_y * y[-1] + (1 - rho_y) * y_bar + alpha * (a[0] - a_bar) + eps_y[x]
end

@parameters colab_likelihood_demo begin
    0 < rho_a < 1
    0 < rho_y < 1
    alpha = 0.4
    a_bar = 1.5
    y_bar = 2.0
    rho_a = 0.8
    rho_y = 0.6
end
"""

model = parse_macro_model(MODEL_SOURCE)
first_order = solve_first_order_model(
    model,
    steady_state_initial_guess={"a": 1.5, "y": 2.0},
    qme_algorithm="schur",
)
assert bool(first_order.solution.converged), "First-order solve did not converge."

observables = ("y", "a")
state_space = build_linear_state_space_from_model(
    model,
    observables,
    first_order_result=first_order,
)
simulation = simulate_linear_gaussian_state_space(
    state_space,
    key=jax.random.PRNGKey(0),
    num_periods=24,
)
steady_lookup = dict(zip(model.timings.var, np.asarray(first_order.steady_state)))
levels = simulation.observations + np.asarray(
    [[steady_lookup[name]] for name in observables],
    dtype=np.float64,
)

theta0 = assemble_parameter_vector(
    model,
    {
        "rho_a": jnp.asarray(0.8, dtype=jnp.float64),
        "rho_y": jnp.asarray(0.6, dtype=jnp.float64),
    },
)

kalman_jit = jax.jit(
    lambda theta: kalman_loglikelihood_from_model_jax(
        model,
        levels,
        observables=observables,
        parameter_values=theta,
        steady_state=first_order.steady_state,
        qme_algorithm="schur",
    )
)

t0 = time.perf_counter()
ll0 = kalman_jit(theta0).block_until_ready()
compile_s = time.perf_counter() - t0
times = []
for _ in range(TIMING_REPS):
    t0 = time.perf_counter()
    kalman_jit(theta0).block_until_ready()
    times.append(time.perf_counter() - t0)

print("Log likelihood:", float(ll0))
print("First JIT call seconds:", compile_s)
print("Steady median seconds:", float(np.median(times)))

theta_alt = assemble_parameter_vector(
    model,
    {
        "rho_a": jnp.asarray(0.33, dtype=jnp.float64),
        "rho_y": jnp.asarray(0.89, dtype=jnp.float64),
    },
)
ll_alt = kalman_jit(theta_alt).block_until_ready()
print("Alternative-parameter log likelihood:", float(ll_alt))
assert not np.isclose(float(ll0), float(ll_alt), rtol=0.0, atol=1e-8), (
    "Likelihood did not change across rho_a/rho_y values; parameter overrides are not reaching the solver."
)

## NumPyro log density and tiny NUTS chain

This is intentionally tiny. It is a functionality check, not posterior inference. Increase samples only after the small chain runs cleanly.

In [ ]:
priors = {
    "rho_a": dist.Uniform(0.05, 0.95),
    "rho_y": dist.Uniform(0.05, 0.95),
}
parameter_samples = {
    "rho_a": jnp.asarray(0.8, dtype=jnp.float64),
    "rho_y": jnp.asarray(0.6, dtype=jnp.float64),
}

log_density_jit = jax.jit(
    lambda: evaluate_numpyro_kalman_log_density_jax(
        model,
        levels,
        priors,
        parameter_samples,
        observables=observables,
        steady_state=first_order.steady_state,
        qme_algorithm="schur",
    )
)
t0 = time.perf_counter()
ld0 = log_density_jit().block_until_ready()
print("NumPyro/JAX log density:", float(ld0), "first call seconds:", time.perf_counter() - t0)

numpyro_model = build_numpyro_kalman_model_jax(
    model,
    levels,
    priors,
    observables=observables,
    steady_state=first_order.steady_state,
    qme_algorithm="schur",
)
kernel = NUTS(numpyro_model, dense_mass=False, target_accept_prob=0.8)
mcmc = MCMC(
    kernel,
    num_warmup=NUTS_WARMUP,
    num_samples=NUTS_SAMPLES,
    num_chains=1,
    progress_bar=True,
)
mcmc.run(jax.random.PRNGKey(123))
samples = mcmc.get_samples()
print({name: np.asarray(value).tolist() for name, value in samples.items()})

## Optional: full Julia CPU vs Python/JAX benchmark

This section is disabled by default because it needs the Julia source repo and can take several minutes. It installs Julia in `/content`, clones the Julia repo, writes a Colab-specific reduced benchmark config, and runs `benchmarks/profile_validation.py`.

Known limitation: on larger DSGE models, NumPyro NUTS can fail if the likelihood differentiation reaches JAX's unsupported QR derivative. The log-density stages still run and the generated report records the failure instead of hiding it.

In [ ]:
if RUN_FULL_JULIA_BENCHMARK:
    if "<your-user-or-org>" in JULIA_REPO_URL:
        raise RuntimeError("Set JULIA_REPO_URL before enabling RUN_FULL_JULIA_BENCHMARK.")

    JULIA_VERSION = "1.12.6"
    julia_dir = Path(f"/content/julia-{JULIA_VERSION}")
    if not julia_dir.exists():
        tarball = f"julia-{JULIA_VERSION}-linux-x86_64.tar.gz"
        url = f"https://julialang-s3.julialang.org/bin/linux/x64/1.12/{tarball}"
        run(["bash", "-lc", f"cd /content && wget -q {url} && tar -xzf {tarball}"])
    os.environ["PATH"] = f"{julia_dir}/bin:" + os.environ["PATH"]
    run(["julia", "--version"])

    julia_root = Path("/content/SurrogateNN_Estimation.jl")
    if not julia_root.exists():
        run(["git", "clone", "--depth", "1", JULIA_REPO_URL, str(julia_root)])

    config_path = ROOT / "benchmarks" / "nn_surrogate_validation_profile.toml"
    report_path = ROOT / "docs" / "benchmark_report_colab.md"
    config_path.write_text(f'''
[global]
upstream_repo = "{julia_root}"
python_executable = "{sys.executable}"
julia_executable = "julia"
thread_count = 2
gate_quantile = 0.8
jax_platform_name = "auto"
report_path = "{report_path}"

[[case]]
name = "small_fs2000"
model_symbol = "FS2000"
model_path = "{julia_root}/models/FS2000.jl"
observables = ["log_gp_obs", "log_gy_obs"]
parameter_subset = ["alp", "bet", "gam"]
numpyro_parameter_subset = ["alp", "gam"]
numpyro_prior_width_scale = 0.01
numpyro_prior_width_floor = 0.0001
numpyro_log_density_reps = 5
numpyro_switching_log_density_reps = 5
numpyro_nuts_warmup = 0
numpyro_nuts_samples = 0
numpyro_nuts_chains = 1
numpyro_target_accept_prob = 0.8
numpyro_seed = 20260712
state_names = ["log_gp_obs", "log_gy_obs"]
periods = 48
shock_seed = 1234
shock_scale_by_exo = {{ e_a = 0.25, e_m = 0.25 }}
measurement_error_scale = 1.0e-8
jitter = 1.0e-9
solve_reps = 1
kalman_value_reps = 10
kalman_per_period_reps = 3
kalman_paths_reps = 1
kalman_grad_reps = 0
gate_stats_reps = 1
switching_fixed_reps = 1
switching_reps = 1
sep_reps = 0
gate_periods = 1
gate_mode = "soft"
gate_beta_eps = 2.0
gate_beta_y = 2.0
gate_hard_threshold = 0.5
gate_prob_floor = 1.0e-4
gate_prob_ceiling = 0.9999
soft_mixture = "logsumexp"
sep_eval_periods = 2
sep_periods = 2
sep_branching_order = 0
sep_nnodes = 3
sep_sparse_tree = true
sep_maxit = 4
sep_tol = 1.0e-5
sep_accept_tol = 1.0e-3
sep_inv_maxit = 2
sep_inv_step_tol = 1.0e-5
sep_inv_resid_tol = 1.0e-5
sep_inv_lambda = 1.0e-4

[[case]]
name = "medium_sw07_hlt"
model_symbol = "Smets_Wouters_2007_HLT"
model_path = "{julia_root}/models/Smets_Wouters_2007_HLT.jl"
observables = ["dy", "dc", "dinve", "labobs", "pinfobs", "dwobs", "robs"]
parameter_subset = ["cprobp", "cindp", "curvp"]
numpyro_parameter_subset = ["cprobp", "cindp"]
numpyro_prior_width_scale = 0.005
numpyro_prior_width_floor = 0.0001
numpyro_log_density_reps = 1
numpyro_switching_log_density_reps = 1
numpyro_nuts_warmup = 0
numpyro_nuts_samples = 0
numpyro_nuts_chains = 1
numpyro_target_accept_prob = 0.8
numpyro_seed = 20260712
state_names = ["dy", "dc", "dinve", "labobs", "pinfobs", "dwobs", "robs"]
periods = 24
shock_seed = 4321
shock_scale_by_exo = {{ ea = 1.0, eb = 1.0, eg = 1.0, em = 1.0, epinf = 1.0, eqs = 1.0, ew = 1.0 }}
measurement_error_scale = 1.0e-8
jitter = 1.0e-9
solve_reps = 1
kalman_value_reps = 1
kalman_per_period_reps = 1
kalman_paths_reps = 0
kalman_grad_reps = 0
gate_stats_reps = 1
switching_fixed_reps = 1
switching_reps = 1
sep_reps = 0
gate_periods = 1
gate_mode = "soft"
gate_beta_eps = 2.0
gate_beta_y = 2.0
gate_hard_threshold = 0.5
gate_prob_floor = 1.0e-4
gate_prob_ceiling = 0.9999
soft_mixture = "logsumexp"
sep_eval_periods = 1
sep_periods = 1
sep_branching_order = 0
sep_nnodes = 3
sep_sparse_tree = true
sep_maxit = 4
sep_tol = 1.0e-5
sep_accept_tol = 1.0e-3
sep_inv_maxit = 2
sep_inv_step_tol = 1.0e-5
sep_inv_resid_tol = 1.0e-5
sep_inv_lambda = 1.0e-4
''')

    run([sys.executable, str(ROOT / "benchmarks" / "profile_validation.py")], cwd=ROOT)
    print(report_path.read_text()[:8000])
else:
    print("Full Julia benchmark skipped. Set RUN_FULL_JULIA_BENCHMARK = True after configuring JULIA_REPO_URL.")

## Optional: Gemini analysis

This uses the official Google Gen AI SDK package `google-genai`. In Colab, add a secret named `GOOGLE_API_KEY`, or set `os.environ["GOOGLE_API_KEY"]`. Keep the prompt factual and ask Gemini to flag uncertainty rather than inventing conclusions.

In [ ]:
if USE_GEMINI:
    try:
        from google.colab import userdata
        api_key = userdata.get("GOOGLE_API_KEY")
    except Exception:
        api_key = None
    api_key = api_key or os.environ.get("GOOGLE_API_KEY")
    if not api_key:
        raise RuntimeError("Set GOOGLE_API_KEY in Colab secrets or os.environ before enabling USE_GEMINI.")

    from google import genai
    client = genai.Client(api_key=api_key)

    report_candidates = [
        ROOT / "docs" / "benchmark_report_colab.md",
        ROOT / "docs" / "benchmark_report_2026-07-12.md",
    ]
    report_text = ""
    for path in report_candidates:
        if path.exists():
            report_text = path.read_text()
            break
    if not report_text:
        report_text = f"Self-contained smoke: loglik={float(ll0)}, log_density={float(ld0)}, devices={jax.devices()}"

    prompt = f"""
You are reviewing a DSGE Python/JAX vs Julia benchmark. Be strict and do not overclaim.
Summarize: (1) what ran, (2) what did not run, (3) whether GPU was actually used,
(4) what the next engineering bottleneck is, and (5) what result would be required
before using this in a paper.

Benchmark/report text:
{report_text[:24000]}
"""
    response = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
    print(response.text)
else:
    print("Gemini disabled. Set USE_GEMINI = True after configuring GOOGLE_API_KEY.")

## Interpretation checklist

- If `jax.devices()` shows only CPU, this is not a GPU benchmark even if Colab runtime says GPU.
- The self-contained NumPyro NUTS chain is only a functionality check.
- The full benchmark report must be read for parity before timing claims are meaningful.
- If full DSGE NUTS fails with a QR derivative error, that is a known blocker in the current first-order state reduction path, not a successful HMC benchmark.